# Lab 03 — Tools: Function Calling & MCP

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

This week the agent gets hands: we open the **ACT** step of the agent loop. Following the
lecture's lab tie-in, you will **wire one calculator tool twice — raw function calling first,
then via a minimal MCP-style server** — and extend the raw wiring into the research agent's
first real toolbox (search, read, compute) over a small offline corpus.

**Learning objectives.** After this lab you can:

- declare a tool as *name + description + JSON Schema* and convert between provider formats,
- implement the **request-execute-append cycle** over `ollama.chat(tools=...)` with a dispatch table,
- treat malformed and failing tool calls as **observations**, with actionable error messages,
- run a **multi-tool conversation** (search → read → compute) on an offline corpus,
- build a **minimal MCP-style server and host** in plain Python and use runtime discovery (`tools/list`),
- locate the **security boundary** of a tool-using agent.

Estimated time: 90–120 minutes. Fill every gap (three consecutive underscores); fold-out
solutions sit below each gapped cell — except for the **report tasks** (R1–R4), which go
into your lab report.

## Theory recap: how agents act on the world

### What a tool call really is

Last week we built the agent loop — reason, act, observe — with the growing message list as
the agent's only state. Today we open the **ACT** step. The term *function calling* is a
misnomer: the model never executes anything. Three things happen instead. **Declaration**:
alongside the messages, the runtime sends a list of tool definitions — each a *name*, a
natural-language *description*, and a *JSON Schema* for the parameters — with **every**
request; the model has no persistent registry of tools. **Emission**: if the model decides a
tool would help, it answers not in prose but with a structured block containing the tool's
name and JSON arguments. **Execution**: your runtime parses, validates and runs the function
— in your process. Emitting well-formed calls is trained behaviour (Toolformer, Schick et
al., 2023; an API primitive since OpenAI, June 2023), and *strict mode* (constrained
decoding) can guarantee schema-valid JSON arguments — though never semantically sensible ones.

### The request-execute-append cycle

One reasoning step is one round trip: **REQUEST** (messages + tool schemas) → **EMIT** (the
model returns a `tool_use` block) → **EXECUTE** (the runtime runs the function) → **APPEND**
(the result joins the messages) — repeated until the model answers in plain text. The stop
signal is the *stop reason*: `tool_use` vs `end_turn`. Each result carries its call's *id* so
the model can pair results with calls — essential for **parallel calls**, where one assistant
turn emits several `tool_use` blocks at once. (Ollama's API, which we use below, has no
explicit call ids and no `tool_choice` parameter; the *absence* of `tool_calls` plays the
role of `end_turn`.) Whether tools get used at all is a per-request runtime parameter, the
**tool-choice mode**: `auto`, `required/any`, a *specific tool*, or `none`.

### Failure as observation

A robust loop treats tool errors as **data, not exceptions**: the failure goes back as an
error-flagged tool result, and the model reads it and adapts. Validate arguments against the
schema *before* executing; cap retries per tool and per run; give every call a timeout; never
blind-retry non-idempotent operations; escalate after $N$ failures. The design rule: an
actionable error says *what was wrong and what a valid call looks like* — "Error 500" teaches
the model nothing.

### The schema is a prompt

Tool definitions ride along in every request, so the model reads every word: `verb_noun`
names (`search_web`, not `doSearch3`), descriptions that say when to use the tool *and when
not*, parameter examples and enums, and concise structured outputs — the model pays tokens to
read whatever you return. Think **ACI**, the *agent-computer interface*: interface design
where the user is a language model under token pressure.

### The Model Context Protocol

Without a standard, $N$ hosts and $M$ systems need $N \times M$ bespoke integrations; a
shared protocol turns this into $N + M$ (Anthropic, 2024). MCP names three roles: the
**host** (the LLM app — owns the model loop and merges tools from many servers), the
**client** (exactly one stateful session per server), and the **server** (any process that
wraps one capability). Servers expose three primitives distinguished by *who controls them*:
**tools** (model-controlled — exactly the function-calling primitive, standardised),
**resources** (application-controlled, URI-addressed, side-effect-free) and **prompts**
(user-controlled templates). Underneath sits JSON-RPC 2.0 over a stateful session, carried by
the *stdio* transport (local subprocess) or *Streamable HTTP* (remote, OAuth 2.1). The
genuinely new capability is **discovery**: `tools/list` returns tool schemas at runtime and
`tools/call` invokes them — a host can learn the tools of a server it has never seen.

### The security boundary

**The model proposes; the runtime disposes.** Every control — validation, allowlists,
confirmation gates — lives runtime-side, and tool results are untrusted input (indirect
prompt injection). One sentence to memorise: *an LLM is exactly as dangerous as the tools its
runtime agrees to execute on its behalf.*

## Part A — Setup & Ollama connectivity

We keep dependencies minimal: `ollama` plus the standard library. The lab uses a local,
tool-capable model via Ollama; any 7–30B model with tool support works.

In [ ]:
import ast
import inspect
import json
import operator
import os
import re
from pathlib import Path

import ollama

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")   # any tool-capable 7-30B model works
DATA_DIR = Path("data")

OLLAMA_OK = False
try:
    ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Reply with: ready"}])
    OLLAMA_OK = True
    print(f"Ollama is reachable and model '{MODEL}' responds.")
except Exception as e:
    print("Could not reach Ollama:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")
    print("You can still work through the non-LLM cells of this notebook.")

In [ ]:
# A quick look at the offline corpus this lab ships with (created by the build script).
NOTES_DIR = DATA_DIR / "notes"
for p in sorted(NOTES_DIR.glob("*.md")):
    print(f"{p.name:32s} {p.stat().st_size:5d} bytes")

SEARCH_INDEX = json.loads((DATA_DIR / "search_index.json").read_text(encoding="utf-8"))
print(f"\nsearch_index.json: {len(SEARCH_INDEX)} snippets, e.g. '{SEARCH_INDEX[0]['title']}'")

## Part B — Declaring the calculator tool

The lecture's slide *"Declaring a tool: the JSON Schema"* showed the calculator declaration;
here you build it for real. Two pieces:

1. a **safe evaluator** — the lecture insisted on *"a safe evaluator rather than Python's
   `eval`"*, because a calculator that evals arbitrary strings is a remote-code-execution
   service with a friendly name;
2. the **tool declaration** — name, description, JSON Schema — plus a converter to the
   OpenAI-style format that Ollama expects (same anatomy, different field names).

In [ ]:
# A safe arithmetic evaluator based on Python's AST - no eval() anywhere.
_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: ___,
    ast.Div: ___,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.FloorDiv: operator.floordiv,
}

def safe_eval(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression (numbers, + - * / ** % //, parentheses)."""
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = _eval(node.operand)
            return value if isinstance(node.op, ast.UAdd) else -value
        if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
            return _ALLOWED_BINOPS[type(node.op)](_eval(node.left), _eval(node.right))
        raise ValueError(
            "Unsupported element in expression - only numbers, the operators "
            "+ - * / ** % // and parentheses are allowed, e.g. '(17.4 - 3) * 2'.")
    try:
        tree = ast.parse(expression, mode=___)   # 'eval' mode: exactly one expression
    except SyntaxError:
        # An ACTIONABLE error: say what was wrong and what a valid call looks like.
        raise ValueError(
            "Could not parse the expression. Provide pure arithmetic without units, "
            "words or symbols like '%of' - e.g. '0.15 * 240' instead of '15% of 240'.")
    return _eval(tree.body)

print(safe_eval("(17.4 - 3) * 2"))   # expected: 28.8
print(safe_eval("2 ** 10 / 4"))      # expected: 256.0

<details>
<summary><b>Click here for the solution</b></summary>

```python
_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.FloorDiv: operator.floordiv,
}
# ... and:
    try:
        tree = ast.parse(expression, mode="eval")   # 'eval' mode: exactly one expression
    except SyntaxError:
        raise ValueError(
            "Could not parse the expression. Provide pure arithmetic without units, "
            "words or symbols like '%of' - e.g. '0.15 * 240' instead of '15% of 240'.")
    return _eval(tree.body)
```

</details>

In [ ]:
# The declaration from the lecture (Anthropic Messages API style).
calculator_tool = {
    "name": ___,
    "description": (
        "Evaluate a basic arithmetic expression and return the numeric result. "
        "Use this for any exact computation instead of estimating."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The expression to evaluate, e.g. '(17.4 - 3) * 2'",
            }
        },
        "required": [___],
    },
}

def to_ollama_tool(tool: dict) -> dict:
    """Convert an Anthropic-style declaration to the OpenAI/Ollama format.
    Same anatomy - name, description, parameter schema - different field names."""
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool[___],
        },
    }

print(json.dumps(to_ollama_tool(calculator_tool), indent=2))

<details>
<summary><b>Click here for the solution</b></summary>

```python
calculator_tool = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression and return the numeric result. "
        "Use this for any exact computation instead of estimating."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The expression to evaluate, e.g. '(17.4 - 3) * 2'",
            }
        },
        "required": ["expression"],
    },
}

def to_ollama_tool(tool: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool["input_schema"],
        },
    }
```

</details>

> **Q:** Why is the term "function calling" a misnomer, and why does the distinction matter?
<details><summary>Click for answer</summary>

The model does not call anything — it emits structured text that *describes* a desired call. Execution happens exclusively in the runtime, which holds the credentials and performs the side effects. The distinction matters for safety and architecture: every security control (validation, allowlists, confirmation gates) must live runtime-side, because that is where action actually occurs — and it explains why a model "with tools" is harmless until someone wires an executor to it.

</details>

## Part C — Wiring 1: raw function calling

Now the first of the two wirings from the lecture's tie-in slide: **you hand-write the JSON
Schema, you maintain a dispatch table mapping tool names to Python functions, and you own the
execute-append loop.** Zero dependencies, full control — every byte that reaches the model is
a byte you put there.

In [ ]:
# The dispatch table: tool name -> Python function. This IS the raw wiring.
TOOL_FUNCTIONS = {"calculator": safe_eval}

def execute_tool_call(name: str, args: dict, tool_functions: dict) -> str:
    """Run one tool call; ALWAYS return a string result - errors included (they are data)."""
    fn = tool_functions.___(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool '{name}'. Available tools: {list(tool_functions)}."})
    try:
        result = fn(___)          # keyword arguments straight from the parsed JSON
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

# Direct smoke test, no LLM involved:
print(execute_tool_call("calculator", {"expression": "(17.4 - 3) * 2"}, TOOL_FUNCTIONS))
print(execute_tool_call("calculator", {"expression": "import os"}, TOOL_FUNCTIONS))

<details>
<summary><b>Click here for the solution</b></summary>

```python
def execute_tool_call(name: str, args: dict, tool_functions: dict) -> str:
    fn = tool_functions.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool '{name}'. Available tools: {list(tool_functions)}."})
    try:
        result = fn(**args)       # keyword arguments straight from the parsed JSON
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})
```

</details>

In [ ]:
def run_agent(user_msg, tools, tool_functions, model=MODEL, max_steps=8,
              executor=None, verbose=True):
    """The request-execute-append cycle from the lecture, wired to Ollama."""
    executor = executor or execute_tool_call
    messages = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        # REQUEST: the conversation so far + the tool schemas go to the model.
        resp = ollama.chat(model=model, messages=messages, tools=___)
        msg = resp["message"]
        messages.append(msg)                      # the assistant turn joins the history
        tool_calls = msg.get("tool_calls")
        # Stop signal: no tool calls = a plain-text answer (Ollama's 'end_turn').
        if not ___:
            if verbose:
                print(f"[step {step}] final answer")
            return msg["content"], messages
        # EXECUTE + APPEND: one result per call, ALL results before the next request.
        for call in tool_calls:
            name = call["function"]["name"]
            args = dict(call["function"]["arguments"] or {})
            result = executor(name, args, tool_functions)
            if verbose:
                print(f"[step {step}] {name}({args}) -> {result[:80]}")
            messages.append({"role": ___, "tool_name": name, "content": result})
    return "(stopped: step limit reached)", messages

<details>
<summary><b>Click here for the solution</b></summary>

```python
def run_agent(user_msg, tools, tool_functions, model=MODEL, max_steps=8,
              executor=None, verbose=True):
    executor = executor or execute_tool_call
    messages = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        resp = ollama.chat(model=model, messages=messages, tools=tools)
        msg = resp["message"]
        messages.append(msg)
        tool_calls = msg.get("tool_calls")
        if not tool_calls:
            if verbose:
                print(f"[step {step}] final answer")
            return msg["content"], messages
        for call in tool_calls:
            name = call["function"]["name"]
            args = dict(call["function"]["arguments"] or {})
            result = executor(name, args, tool_functions)
            if verbose:
                print(f"[step {step}] {name}({args}) -> {result[:80]}")
            messages.append({"role": "tool", "tool_name": name, "content": result})
    return "(stopped: step limit reached)", messages
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The loop is the lecture's flow slide, line by line. **REQUEST**: `ollama.chat` receives the
full message list *plus* the tool schemas — the schemas travel with every request, because
the model keeps no registry. **EMIT**: the response either contains `tool_calls` (the
structured blocks) or it does not; that presence/absence is Ollama's version of the
`tool_use` / `end_turn` stop reason. **EXECUTE**: the dispatch table maps the emitted name to
a Python function; the parsed JSON arguments are passed as keyword arguments. **APPEND**: the
result is added as a `role: "tool"` message. Note that we append the assistant turn *and* one
result per call before the next request — that is the contract for parallel calls (several
`tool_calls` in one turn). Ollama pairs results by order and `tool_name`; the Anthropic and
OpenAI APIs use explicit call ids for the same purpose. `max_steps` is the step limit from
Session 02 — without it, a failing tool becomes an infinite loop.

</details>

In [ ]:
# First end-to-end run: the model routes arithmetic through YOUR code.
try:
    answer, transcript = run_agent(
        "What is (17.4 - 3) * 2 - 121 / 11? Use the calculator; do not estimate.",
        tools=[___],
        tool_functions=TOOL_FUNCTIONS,
    )
    print("\n" + str(answer))
    print(f"\nTranscript: {len(transcript)} messages "
          "(user, assistant+tool_calls, tool result, final assistant)")
except Exception as e:
    print("Ollama not reachable - start `ollama serve` first. Details:", e)

<details>
<summary><b>Click here for the solution</b></summary>

```python
    answer, transcript = run_agent(
        "What is (17.4 - 3) * 2 - 121 / 11? Use the calculator; do not estimate.",
        tools=[to_ollama_tool(calculator_tool)],
        tool_functions=TOOL_FUNCTIONS,
    )
```

</details>

> **Q:** Walk through the request-execute-append cycle and name the signal that terminates it.
<details><summary>Click for answer</summary>

Request: messages plus tool schemas go to the API. Emit: the model returns either plain text or tool-call blocks. Execute: the runtime runs the named function with the parsed arguments. Append: the result is added to the message list, and the cycle repeats. The loop terminates when the stop reason is `end_turn` (a plain-text answer) rather than `tool_use` — in Ollama's API, when the response carries no `tool_calls`.

</details>

> **Q:** How do tool-call ids work and why are they necessary?
<details><summary>Click for answer</summary>

Each `tool_use` block the model emits carries a unique id; the runtime must attach the same id to the corresponding tool-result message, and the model matches results to calls by id on the next request. This is essential for parallel calls, where several calls are outstanding in one turn and results would otherwise be ambiguous. (Ollama omits explicit ids and pairs results by order and tool name — the concept is the same, the guarantee weaker.)

</details>

> **📝 Report task R1:** Name the four tool-choice modes from the lecture and give one
> concrete situation in our research agent where each mode is the right choice. Ollama's
> Python API exposes no `tool_choice` parameter — state which mode its default behaviour
> corresponds to.
>
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part D — Failure as observation: validation, errors, self-correction

In production, tools fail constantly. The lecture's rule: **an error is not an exception to
crash on — it is an observation to feed back.** Here you (1) validate arguments against the
schema *before* executing ("reject, do not crash"), (2) write error messages a model can act
on, and (3) watch the model self-correct when it receives one.

In [ ]:
JSON_TO_PY = {"string": str, "number": (int, float), "integer": int, "boolean": bool}

def validate_args(args: dict, schema: dict):
    """Check args against the input schema. Return an ACTIONABLE message, or None if OK."""
    props = schema["properties"]
    for field in schema.get("required", []):
        if field not in ___:
            hint = props[field].get("description", "")
            return (f"Missing required parameter '{field}'. "
                    f"A valid call provides: {list(props)}. Hint: {hint}")
    for field, value in args.items():
        if field not in props:
            return f"Unknown parameter '{field}'. Valid parameters: {list(props)}."
        expected = JSON_TO_PY.get(props[field]["type"], object)
        if not isinstance(value, ___):
            return (f"Parameter '{field}' must be of type {props[field]['type']}, "
                    f"got {type(value).__name__}.")
    return None

TOOL_SCHEMAS = {"calculator": calculator_tool["input_schema"]}

def execute_tool_call_v2(name: str, args: dict, tool_functions: dict, tool_schemas: dict) -> str:
    """Validate first, execute second - failures become observations, not exceptions."""
    fn = tool_functions.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool '{name}'. Available: {list(tool_functions)}."})
    problem = ___(args, tool_schemas[name])
    if problem is not None:
        return json.dumps({"error": problem})          # reject cheaply, do not crash
    try:
        return json.dumps({"result": fn(**args)})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

<details>
<summary><b>Click here for the solution</b></summary>

```python
def validate_args(args: dict, schema: dict):
    props = schema["properties"]
    for field in schema.get("required", []):
        if field not in args:
            hint = props[field].get("description", "")
            return (f"Missing required parameter '{field}'. "
                    f"A valid call provides: {list(props)}. Hint: {hint}")
    for field, value in args.items():
        if field not in props:
            return f"Unknown parameter '{field}'. Valid parameters: {list(props)}."
        expected = JSON_TO_PY.get(props[field]["type"], object)
        if not isinstance(value, expected):
            return (f"Parameter '{field}' must be of type {props[field]['type']}, "
                    f"got {type(value).__name__}.")
    return None

# ... and inside execute_tool_call_v2:
    problem = validate_args(args, tool_schemas[name])
```

</details>

In [ ]:
# Provoke every failure class directly - no LLM needed. Read each message and ask yourself:
# could a model fix its next call from this text alone?
print(execute_tool_call_v2("calculator", {}, TOOL_FUNCTIONS, TOOL_SCHEMAS))                 # missing param
print(execute_tool_call_v2("calculator", {"expression": ___}, TOOL_FUNCTIONS, TOOL_SCHEMAS))  # wrong type (try 42)
print(execute_tool_call_v2("calculator", {"expression": "17.4kg - 3"}, TOOL_FUNCTIONS, TOOL_SCHEMAS))  # bad content
print(execute_tool_call_v2(___, {"x": 1}, TOOL_FUNCTIONS, TOOL_SCHEMAS))                    # unknown tool (try 'doSearch3')

<details>
<summary><b>Click here for the solution</b></summary>

```python
print(execute_tool_call_v2("calculator", {}, TOOL_FUNCTIONS, TOOL_SCHEMAS))
print(execute_tool_call_v2("calculator", {"expression": 42}, TOOL_FUNCTIONS, TOOL_SCHEMAS))
print(execute_tool_call_v2("calculator", {"expression": "17.4kg - 3"}, TOOL_FUNCTIONS, TOOL_SCHEMAS))
print(execute_tool_call_v2("doSearch3", {"x": 1}, TOOL_FUNCTIONS, TOOL_SCHEMAS))
```

</details>

In [ ]:
# Self-correction: hand the model a transcript in which its call JUST failed,
# and check whether the actionable error message leads to a corrected call.
bad_expr = "15% of 240"
bad_result = execute_tool_call_v2("calculator", {"expression": bad_expr}, ___, TOOL_SCHEMAS)
print("Observation the model will see:", bad_result, "\n")

messages = [
    {"role": "user", "content": "What is 15 percent of 240? Use the calculator."},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "calculator", "arguments": {"expression": bad_expr}}}]},
    {"role": ___, "tool_name": "calculator", "content": bad_result},
]
try:
    resp = ollama.chat(model=MODEL, messages=messages, tools=[to_ollama_tool(calculator_tool)])
    follow_up = resp["message"].get("tool_calls")
    if follow_up:
        print("Self-corrected call:", follow_up[0]["function"]["arguments"])
    else:
        print("Model answered directly:", resp["message"]["content"])
except Exception as e:
    print("Ollama not reachable:", e)

<details>
<summary><b>Click here for the solution</b></summary>

```python
bad_result = execute_tool_call_v2("calculator", {"expression": bad_expr},
                                  TOOL_FUNCTIONS, TOOL_SCHEMAS)
# ...
    {"role": "tool", "tool_name": "calculator", "content": bad_result},
```

</details>

> **Q:** What makes an error message "actionable" for a model? Give a good and a bad example.
<details><summary>Click for answer</summary>

It must state what was wrong and what a valid call looks like, because the message is the only feedback signal the model gets. Bad: "Error 500" — the model learns nothing and will typically retry the identical call. Good: "Invalid date format; expected ISO 8601, e.g. 2026-06-12" — the next call is usually correct. The error message is part of the tool's interface and deserves the same design care as the description.

</details>

> **Q:** Why is idempotency the dividing line for retry policy?
<details><summary>Click for answer</summary>

Retrying an idempotent operation (a search, a read) changes nothing if the first attempt actually succeeded; retrying a non-idempotent one (payment, send-email, delete) after a timeout may execute it twice, because a timeout does not tell you whether the first attempt completed. Mitigations: idempotency keys, check-then-act patterns, human confirmation for such tools — and never blind-retrying writes.

</details>

## Part E — Multi-tool conversations: the research agent's toolbox

The running example needs *web search, a page fetcher and a file writer*. At lab time we stay
offline, so `search_notes` searches a saved snippet collection (our stand-in for web search)
and `read_file` fetches from a small local corpus — with an **allowlist**, because
model-supplied paths are untrusted input. Note how both follow the lecture's *strong
interface* pattern: `verb_noun` names, when-to-use descriptions, bounded parameters, and
concise structured results (title/URL/snippet — never raw HTML).

In [ ]:
read_file_tool = {
    "name": "read_file",
    "description": (
        "Read one note from the local research corpus and return its full text. "
        "Use it AFTER search_notes has identified a relevant file; do not guess file names."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "Name of a corpus file, e.g. 'eu_battery_regulation.md'",
            }
        },
        "required": ["filename"],
    },
}

def read_file(filename: str) -> str:
    """Allowlisted read - never trust model-supplied paths (no traversal, no absolute paths)."""
    allowed = {p.name: p for p in NOTES_DIR.glob("*.md")}
    if filename not in ___:
        raise ValueError(f"'{filename}' is not in the corpus. Available files: {sorted(allowed)}")
    return allowed[filename].read_text(encoding="utf-8")

search_notes_tool = {
    "name": "search_notes",
    "description": (
        "Keyword search over a small offline snippet collection (this lab's stand-in for "
        "web search). Use it to find which sources exist; it returns title, url and snippet "
        "per hit. Fetch full local notes with read_file afterwards. Not for arithmetic."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string",
                      "description": "Search terms, e.g. 'recycling efficiency target'"},
            "max_results": {"type": "integer",
                            "description": "How many hits to return (1-5). Default: 3"},
        },
        "required": ["query"],
    },
}

def search_notes(query: str, max_results: int = 3) -> list:
    """Rank snippets by keyword overlap; return concise, structured results only."""
    max_results = max(1, min(int(max_results), 5))     # enforce the documented bound
    terms = set(re.findall(r"[a-z0-9/]+", query.lower()))
    scored = []
    for entry in SEARCH_INDEX:
        haystack = " ".join([entry["title"], entry["snippet"], *entry["keywords"]]).lower()
        score = sum(1 for t in terms if t in ___)
        if score > 0:
            scored.append((score, entry))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [{"title": e["title"], "url": e["url"], "snippet": e["snippet"]}
            for _, e in scored[:max_results]]

# The grown toolbox - registry, dispatch table, schema table:
TOOLS = [calculator_tool, read_file_tool, search_notes_tool]
TOOL_FUNCTIONS = {"calculator": safe_eval, "read_file": ___, "search_notes": ___}
TOOL_SCHEMAS = {t["name"]: t["input_schema"] for t in TOOLS}
OLLAMA_TOOLS = [to_ollama_tool(t) for t in TOOLS]

print(json.dumps(search_notes("recycling efficiency target lithium"), indent=2)[:400])

<details>
<summary><b>Click here for the solution</b></summary>

```python
def read_file(filename: str) -> str:
    allowed = {p.name: p for p in NOTES_DIR.glob("*.md")}
    if filename not in allowed:
        raise ValueError(f"'{filename}' is not in the corpus. Available files: {sorted(allowed)}")
    return allowed[filename].read_text(encoding="utf-8")

def search_notes(query: str, max_results: int = 3) -> list:
    max_results = max(1, min(int(max_results), 5))
    terms = set(re.findall(r"[a-z0-9/]+", query.lower()))
    scored = []
    for entry in SEARCH_INDEX:
        haystack = " ".join([entry["title"], entry["snippet"], *entry["keywords"]]).lower()
        score = sum(1 for t in terms if t in haystack)
        if score > 0:
            scored.append((score, entry))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [{"title": e["title"], "url": e["url"], "snippet": e["snippet"]}
            for _, e in scored[:max_results]]

TOOL_FUNCTIONS = {"calculator": safe_eval, "read_file": read_file, "search_notes": search_notes}
```

</details>

In [ ]:
# A miniature research-agent episode: search -> read -> compute -> answer.
task = (
    "You are a research assistant with tools. Find out which note in the corpus states the "
    "recycling efficiency target for lithium-based batteries by end of 2025, read that note, "
    "and then use the calculator to work out how many tonnes that target corresponds to for "
    "500,000 tonnes of collected lithium-based batteries. Answer in two sentences and name "
    "the note's file name."
)
try:
    answer, transcript = run_agent(task, tools=___, tool_functions=TOOL_FUNCTIONS,
        executor=lambda n, a, tf: execute_tool_call_v2(n, a, tf, ___),
        max_steps=10)
    print("\n" + str(answer))
except Exception as e:
    print("Ollama not reachable - start `ollama serve` first. Details:", e)

<details>
<summary><b>Click here for the solution</b></summary>

```python
    answer, transcript = run_agent(task, tools=OLLAMA_TOOLS, tool_functions=TOOL_FUNCTIONS,
        executor=lambda n, a, tf: execute_tool_call_v2(n, a, tf, TOOL_SCHEMAS),
        max_steps=10)
```

</details>

> **Q:** Why should a search tool not return the raw HTML of result pages?
<details><summary>Click for answer</summary>

The model pays tokens to read every byte returned, and raw HTML is dominated by markup, navigation and boilerplate irrelevant to the decision at hand — often tens of thousands of tokens. Returning structured minimal fields (title, URL, snippet) can be a hundred-fold token reduction, keeps the context within budget, and gives the model exactly what its next decision (which page to fetch) requires. Output shape is half the interface.

</details>

> **📝 Report task R2:** Tool results are untrusted input. For the multi-tool agent you just
> ran, describe one concrete **indirect prompt injection**: what would an attacker place
> where (think of the snippet collection or a corpus file), what could happen when the agent
> reads it, and which **two runtime-side controls** would blunt the attack?
>
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part F — Wiring 2: a minimal MCP-style server and host

The second wiring from the tie-in slide. A real MCP server speaks **JSON-RPC 2.0** over the
*stdio* or *Streamable HTTP* transport; ours is **in-process** — same message shapes, no
plumbing — so you can see the protocol itself: the `initialize` handshake, **discovery** via
`tools/list`, invocation via `tools/call`, and errors as data (`isError`). Like FastMCP from
the lecture slide, the server derives each tool's schema from the **function signature and
docstring** — the hand-written JSON Schema from Part B disappears.

In [ ]:
class MiniMCPServer:
    """A minimal in-process sketch of an MCP server: JSON-RPC dict in, JSON-RPC dict out.
    Real servers speak exactly these shapes over stdio or Streamable HTTP."""

    PY_TO_JSON = {str: "string", float: "number", int: "integer", bool: "boolean"}

    def __init__(self, name: str):
        self.name = name
        self._tools = {}

    def tool(self, fn):
        """Register a function; the schema is derived from signature + docstring (like FastMCP)."""
        self._tools[fn.__name__] = fn
        return fn

    def _schema_for(self, fn) -> dict:
        props, required = {}, []
        for param in inspect.signature(fn).parameters.values():
            props[param.name] = {"type": self.PY_TO_JSON.get(param.annotation, ___)}
            if param.default is inspect.Parameter.empty:
                required.append(param.name)
        return {"name": fn.__name__,
                "description": inspect.getdoc(fn) or "",
                "inputSchema": {"type": "object", "properties": props, "required": required}}

    def handle(self, request: dict) -> dict:
        """Dispatch one JSON-RPC 2.0 request - the server's entire public surface."""
        method, rid = request["method"], request.get("id")
        params = request.get("params", {})
        if method == "initialize":
            result = {"protocolVersion": "2025-06-18",
                      "serverInfo": {"name": self.name},
                      "capabilities": {"tools": {}}}
        elif method == ___:                                    # discovery
            result = {"tools": [self._schema_for(fn) for fn in self._tools.values()]}
        elif method == "tools/call":
            fn = self._tools.get(params["name"])
            if fn is None:
                return {"jsonrpc": "2.0", "id": rid,
                        "error": {"code": -32601, "message": f"Unknown tool '{params['name']}'"}}
            try:
                output = fn(**params.get("arguments", {}))
                result = {"content": [{"type": "text", "text": str(output)}], "isError": ___}
            except Exception as e:
                result = {"content": [{"type": "text", "text": f"{type(e).__name__}: {e}"}],
                          "isError": True}                     # errors are data, on the wire too
        else:
            return {"jsonrpc": "2.0", "id": rid,
                    "error": {"code": -32601, "message": f"Unknown method '{method}'"}}
        return {"jsonrpc": "2.0", "id": rid, "result": result}


calc_server = MiniMCPServer("calc")

@calc_server.tool
def calculator(expression: str) -> float:
    """Evaluate a basic arithmetic expression and return the numeric result.
    Use this for any exact computation instead of estimating."""
    return safe_eval(expression)

print(json.dumps(calc_server._schema_for(calculator), indent=2))

<details>
<summary><b>Click here for the solution</b></summary>

```python
            props[param.name] = {"type": self.PY_TO_JSON.get(param.annotation, "string")}
# ...
        elif method == "tools/list":                           # discovery
            result = {"tools": [self._schema_for(fn) for fn in self._tools.values()]}
# ...
                output = fn(**params.get("arguments", {}))
                result = {"content": [{"type": "text", "text": str(output)}], "isError": False}
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

Three methods make up the whole protocol sketch. `initialize` is the lifecycle handshake:
client and server agree on a protocol revision (we echo `2025-06-18`, the current spec) and
declare capabilities. `tools/list` is **discovery** — the genuinely new capability compared
to raw function calling: the host learns the tool schemas *at runtime*, generated here from
`inspect.signature` (type hints → JSON types, defaults → not required) and `inspect.getdoc`
(docstring → description). The interface contract from Part B has not disappeared — it moved
into the signature, where tooling checks it and it cannot drift out of sync with the
implementation. The docstring is still a prompt: write it with the same care. `tools/call`
executes and wraps the outcome in a `content` list with an `isError` flag — the wire-level
version of "failure as observation". JSON-RPC error objects (code −32601) are reserved for
*protocol*-level faults such as an unknown method, mirroring the exception/observation split
from Part D.

</details>

In [ ]:
class MiniMCPHost:
    """The host owns the model loop. One 'client' = one session dict per connected server."""

    def __init__(self, model: str = MODEL):
        self.model = model
        self.sessions = []                       # one stateful session per server

    def _rpc(self, server, method, params=None):
        session = next(s for s in self.sessions if s["server"] is server)
        session["next_id"] += 1
        request = {"jsonrpc": "2.0", "id": session["next_id"],
                   "method": method, "params": params or {}}
        response = server.handle(request)        # in-process 'transport'
        if "error" in response:
            return {"isError": True,
                    "content": [{"type": "text", "text": response["error"]["message"]}]}
        return response["result"]

    def connect(self, server):
        """initialize handshake + first discovery - the host learns tools at runtime."""
        self.sessions.append({"server": server, "next_id": 0})
        info = self._rpc(server, ___)
        listed = self._rpc(server, ___)          # runtime discovery
        print(f"Connected to '{info['serverInfo']['name']}' "
              f"(protocol {info['protocolVersion']}): tools = "
              f"{[t['name'] for t in listed['tools']]}")

    def _discover_ollama_tools(self):
        tools, routing = [], {}
        for session in self.sessions:
            listed = self._rpc(session["server"], "tools/list")
            for t in listed["tools"]:
                tools.append({"type": "function", "function": {
                    "name": t["name"], "description": t["description"],
                    "parameters": t["inputSchema"]}})
                routing[t["name"]] = session["server"]
        return tools, routing

    def run(self, user_msg: str, max_steps: int = 8, verbose=True):
        tools, routing = self._discover_ollama_tools()   # fresh discovery each run
        messages = [{"role": "user", "content": user_msg}]
        for step in range(1, max_steps + 1):
            resp = ollama.chat(model=self.model, messages=messages, tools=tools)
            msg = resp["message"]
            messages.append(msg)
            tool_calls = msg.get("tool_calls")
            if not tool_calls:
                return msg["content"], messages
            for call in tool_calls:
                name = call["function"]["name"]
                args = dict(call["function"]["arguments"] or {})
                server = routing.get(name)
                if server is None:
                    text = f"Unknown tool '{name}'."
                else:
                    result = self._rpc(server, ___, {"name": name, "arguments": args})
                    text = result["content"][0]["text"]
                if verbose:
                    print(f"[step {step}] {name}({args}) -> {text[:60]}")
                messages.append({"role": "tool", "tool_name": name, "content": text})
        return "(stopped: step limit reached)", messages


host = MiniMCPHost()
host.connect(calc_server)

<details>
<summary><b>Click here for the solution</b></summary>

```python
    def connect(self, server):
        self.sessions.append({"server": server, "next_id": 0})
        info = self._rpc(server, "initialize")
        listed = self._rpc(server, "tools/list")   # runtime discovery
        print(f"Connected to '{info['serverInfo']['name']}' "
              f"(protocol {info['protocolVersion']}): tools = "
              f"{[t['name'] for t in listed['tools']]}")
# ... and inside run():
                    result = self._rpc(server, "tools/call", {"name": name, "arguments": args})
```

</details>

In [ ]:
# Look at the raw wire messages once - this is all that MCP traffic is.
request = {"jsonrpc": "2.0", "id": 99, "method": ___,
           "params": {"name": "calculator", "arguments": {"expression": "(17.4 - 3) * 2"}}}
print(json.dumps(calc_server.handle(request), indent=2))

# A hostile 'expression' comes back as data with isError=True - not as a crash:
bad = {"jsonrpc": "2.0", "id": 100, "method": "tools/call",
       "params": {"name": "calculator", "arguments": {"expression": "__import__('os')"}}}
print(json.dumps(calc_server.handle(bad), indent=2))

# And the same model behaviour as in Part C - only the integration changed:
try:
    answer, _ = host.run("What is 3.2 * (14 - 5.5)? Use the calculator.")
    print("\n" + str(answer))
except Exception as e:
    print("Ollama not reachable:", e)

<details>
<summary><b>Click here for the solution</b></summary>

```python
request = {"jsonrpc": "2.0", "id": 99, "method": "tools/call",
           "params": {"name": "calculator", "arguments": {"expression": "(17.4 - 3) * 2"}}}
```

</details>

> **📝 Report task R3:** Complete the cell below. Add a second tool, `unit_convert`, to
> `calc_server` — supported conversions at least kg↔g, km↔m, h↔min, with an *actionable*
> error for unsupported pairs and a docstring written as carefully as a tool description
> (it *is* one). Then demonstrate that the **host code stays unchanged**: a fresh host picks
> the new tool up via `tools/list` discovery alone. In your report, contrast this with what
> the same extension would require in the raw wiring of Part C.
>
> *No solution is provided — include your code and a short justification in your lab report.*

In [ ]:
# 📝 Report task R3 - complete this cell (no fold-out solution below!).

@calc_server.tool
def unit_convert(value: float, from_unit: str, to_unit: str) -> float:
    """___"""
    factors = {("kg", "g"): 1000.0, ___}
    ___
    ___

host2 = MiniMCPHost()
host2.connect(calc_server)      # discovery does the rest - nothing else changes
try:
    answer, _ = host2.run("Convert 2.5 kg to grams, then add 300 g. Use your tools.")
    print("\n" + str(answer))
except Exception as e:
    print("Ollama not reachable:", e)

> **Q:** Name MCP's three server primitives and who controls each. Why does the control model matter?
<details><summary>Click for answer</summary>

**Tools** are model-controlled (the LLM decides to invoke them; they may have side effects). **Resources** are application-controlled (URI-addressed readable data the host chooses to attach; side-effect-free). **Prompts** are user-controlled (templates invoked explicitly, e.g. slash commands). The control model encodes distinct trust postures — "model may act at will", "application deliberately provides context" and "user explicitly triggers" are different authorisation decisions, and the protocol keeps them separate rather than collapsing everything into tools.

</details>

> **Q:** Why did MCP succeed where earlier attempts like ChatGPT plugins faltered? *(not exam-relevant)*
<details><summary>Click for answer</summary>

Plugins were single-vendor, tied to one consumer product, with the host controlling distribution. MCP is an open specification any host can implement: building a server targets every client at once (the $N+M$ economics), local stdio servers made experimentation trivial, and rapid adoption by competing vendors (OpenAI, Google, Microsoft in 2025) removed the lock-in fear. Timing helped too: by 2025 agents were a real workload, so demand for integrations existed.

</details>

> **📝 Report task R4:** You built the same calculator twice (Parts C and F). State the
> **decision rule**: when is raw function calling the better engineering choice, and when
> does MCP pay off? Name concretely which code *disappeared* with MCP and which *costs
> appeared* — based on your own two implementations.
>
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part G — Tuning & exploration

No gaps here — this part is for experiments. The most instructive knob is the first one: the
lecture claimed that *"the difference in end-to-end task success between a weak and a strong
interface is routinely larger than the difference between model generations."* Test it.

1. **`description_quality`** — `"weak"` turns the calculator into the lecture's weak
   interface (`doCalc3`, description *"performs calculation"*, undocumented parameter).
   Does the model still route arithmetic through the tool? Does it still fill `expression`
   correctly?
2. **`temperature`** — higher values make argument emission less reliable.
3. **`max_steps`** — tighten the budget on the Part E multi-tool task and watch it hit the limit.
4. **`model`** — any tool-capable Ollama model, e.g. `llama3.1:8b` or `qwen2.5:14b`.

In [ ]:
TUNING = {
    "description_quality": "strong",   # "strong" or "weak"
    "temperature": 0.2,                # try 0.0 ... 1.5
    "max_steps": 6,
    "model": MODEL,
}

def tuned_calculator_tool():
    tool = json.loads(json.dumps(calculator_tool))            # deep copy
    if TUNING["description_quality"] == "weak":
        tool["name"] = "doCalc3"
        tool["description"] = "performs calculation"
        tool["input_schema"]["properties"]["expression"].pop("description", None)
    return tool

probe_tasks = [
    "What is 987 * 123?",
    "Compute (2.5 + 7.5) ** 2 / 4.",
    "A battery pack has 96 cells of 3.6 V in series. What is the pack voltage?",
]

def tuning_run():
    tool = tuned_calculator_tool()
    functions = {tool["name"]: safe_eval}
    print(f"interface={TUNING['description_quality']!r}, "
          f"temperature={TUNING['temperature']}, model={TUNING['model']!r}\n")
    for task in probe_tasks:
        try:
            resp = ollama.chat(model=TUNING["model"],
                               messages=[{"role": "user", "content": task}],
                               tools=[to_ollama_tool(tool)],
                               options={"temperature": TUNING["temperature"]})
            calls = resp["message"].get("tool_calls")
            args = dict(calls[0]["function"]["arguments"]) if calls else None
            print(f"{task[:52]:54s} tool used: {bool(calls)!s:5s} args: {args}")
        except Exception as e:
            print("Ollama not reachable - start `ollama serve` first. Details:", e)
            break

tuning_run()

In [ ]:
# Optional interactive controls (the notebook works fine without ipywidgets).
try:
    import ipywidgets as widgets
    from IPython.display import display

    dd_desc = widgets.Dropdown(options=["strong", "weak"], value=TUNING["description_quality"],
                               description="interface")
    sl_temp = widgets.FloatSlider(value=TUNING["temperature"], min=0.0, max=1.5, step=0.1,
                                  description="temperature")
    btn = widgets.Button(description="Run probes")

    def _on_click(_):
        TUNING["description_quality"] = dd_desc.value
        TUNING["temperature"] = sl_temp.value
        tuning_run()

    btn.on_click(_on_click)
    display(dd_desc, sl_temp, btn)
except ImportError:
    print("ipywidgets not installed - edit the TUNING dict above and re-run tuning_run().")

## Wrap-up

**Takeaways.**

- A tool call is **structured text**: the model emits a request; your runtime executes it and
  appends the result — one stateless round trip per step, terminated by the stop signal.
- **Failures are observations**: validate before executing, return actionable error messages,
  cap retries, protect non-idempotent operations — and watch the model self-correct.
- **The schema is a prompt** (ACI): names, descriptions and output shape bought you more
  reliability in Part G than any model swap would.
- **MCP standardises where the functions come from**, not what they are: host/client/server,
  JSON-RPC, and runtime discovery via `tools/list` — the request-execute-append cycle itself
  was identical in both wirings.
- **The model proposes; the runtime disposes** — every security control lives runtime-side,
  and tool results are untrusted input.

**Next week (Session 04):** prompt engineering for agents — system prompts, few-shot
examples, and chain-of-thought. The tool descriptions you wrote today were already prompts;
next week we design the rest of the context with the same discipline.

---

### 📋 For your lab report

| # | Task | Where |
|---|------|-------|
| R1 | Four tool-choice modes + one research-agent use case each; Ollama's default mode | after Part C |
| R2 | One concrete indirect prompt injection on the Part E agent + two runtime-side controls | after Part E |
| R3 | `unit_convert` on the MCP server (code) + why the host stays unchanged vs. Part C | Part F |
| R4 | Decision rule raw function calling vs. MCP; what disappeared, what it cost | after Part F |

*No fold-out solutions exist for these — write them up in your own words/code.*